# Shift-aware CRC 3D — M&Ms cardiaque (Kaggle)
Accélérateur : **GPU T4 x2**. Inputs : le dataset de code `shift-crc-3d` et le dataset **privé** `mnms-npz` (données déjà prétraitées sur le PC avec `scripts/preprocess_mnms.py`). Optionnel : la sortie d'une version précédente de ce notebook (folds déjà faits).

**Données M&Ms : dataset Kaggle PRIVÉ uniquement (accord d'utilisation, clause 5).**

Après tout redémarrage de session, relancer les cellules 1 et 2.

In [ ]:
!pip install -q --no-deps nibabel
import numpy, torch
print('numpy', numpy.__version__, '| torch', torch.__version__, '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU  <-- activer le GPU dans Settings')

In [ ]:
import os, shutil, glob
import pandas as pd

AUG = 'strong'
RUNS = f'runs_mnms_{AUG}'
W = '/kaggle/working'

hits = [h for h in glob.glob('/kaggle/input/**/scripts/train_seg.py', recursive=True) if '--aug' in open(h).read()]
assert hits, "Code (version récente) introuvable : ajouter / mettre à jour le dataset shift-crc-3d"
src = hits[0].rsplit('/scripts/', 1)[0]
allmeta = glob.glob('/kaggle/input/**/metadata.csv', recursive=True)
for m in allmeta: print('metadata trouvé :', m, '| colonnes :', open(m).readline().strip())
meta = [m for m in allmeta if 'patient' in open(m).readline()]
if not meta:
    print('\nContenu de /kaggle/input (3 niveaux) :')
    for root, dirs, files in os.walk('/kaggle/input'):
        if root.count('/') <= 6:
            print(root, '|', len(files), 'fichiers', '|', files[:3])
assert meta, "Données M&Ms prétraitées introuvables : voir la liste ci-dessus"
# the .npz files may be split over several zips / sub-folders (part1, part2, ...):
# gather them all (symbolic links, no copy) in one working folder next to metadata.csv
ds_root = '/'.join(meta[0].split('/')[:6])          # /kaggle/input/datasets/<user>/<dataset>
NPZ = f'{W}/mnms_npz'
os.makedirs(NPZ, exist_ok=True)
for f in glob.glob(f'{ds_root}/**/*.npz', recursive=True) + [meta[0]]:
    dst = f'{NPZ}/{os.path.basename(f)}'
    if not os.path.exists(dst):
        os.symlink(f, dst)
md_ = pd.read_csv(meta[0])
missing = [n for n in md_.name if not os.path.exists(f'{NPZ}/{n}.npz')]
print('fichiers .npz réunis :', len(glob.glob(f'{NPZ}/*.npz')), '| manquants :', len(missing), missing[:5])
assert not missing, "Des volumes listés dans metadata.csv manquent : vérifier que les 3 zips sont bien uploadés"
SITES = sorted(md_.site.unique())
prev = glob.glob(f'/kaggle/input/**/{RUNS}', recursive=True)
PREV = prev[0] if prev else None

print('CODE :', src); print('NPZ  :', NPZ, '|', len(md_), 'cas,', md_.patient.nunique(), 'patients')
print(md_.groupby(['site', 'vendor']).patient.nunique())
print('SITES :', SITES, '| PREV :', PREV)
shutil.copytree(src, f'{W}/code', dirs_exist_ok=True)
os.chdir(f'{W}/code')
if PREV:
    shutil.copytree(PREV, f'{W}/{RUNS}', dirs_exist_ok=True)

## Entraînement + inférence de chaque centre cible (leave-one-centre-out)

In [ ]:
for SITE in SITES:
    RUN = f'{W}/{RUNS}/{SITE}'
    if os.path.exists(f'{RUN}/summaries.npz'):
        print(f'== {SITE} : déjà fait, sauté'); continue
    print(f'\n================ FOLD {SITE} ================')
    !python scripts/train_seg.py --data {NPZ} --target_site {SITE} --out {RUN} --iters 8000 --aug {AUG} --patch 16 128 128 --batch 4 --max_hours 1.8
    if not os.path.exists(f'{RUN}/best.pt'):
        raise RuntimeError(f'entraînement de {SITE} échoué : voir le message ci-dessus')
    !python scripts/infer_summaries.py --data {NPZ} --run {RUN} --mc 8

## Analyse conforme (par phase cardiaque : un cas par patient)

In [ ]:
for PH in ['ED', 'ES']:
    !python scripts/run_v2.py --runs "{W}/{RUNS}/*" --out {W}/results_mnms/{PH} --phase {PH} --reps 300
import zipfile
with zipfile.ZipFile(f'{W}/summaries_mnms.zip', 'w') as z:
    for f in sorted(glob.glob(f'{W}/{RUNS}/*/summaries.npz')):
        z.write(f, f.split(f'{RUNS}/')[1])
print('zip :', f'{W}/summaries_mnms.zip')

In [ ]:
os.chdir(W)
shutil.rmtree(f'{W}/code', ignore_errors=True)
!du -sh {W}/*